# Detección de fraudes con Machine Learning
**Pasos del taller:**
1. Generar aleatoriamente un archivo con: monto (COP), hora del día, distancia a la residencia, ingreso mensual y fraude (sí/no).
2. Revisarlo y hacerlo más realista.
3. Encontrar un modelo que determine un posible fraude.

In [ ]:
# ============================================================
# 1. IMPORTAR LIBRERÍAS
# ============================================================
import numpy as np                      # cálculos numéricos y números aleatorios
import pandas as pd                     # manejo de tablas (DataFrames)
import matplotlib.pyplot as plt         # gráficas

from sklearn.model_selection import train_test_split, cross_val_predict, StratifiedKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.metrics import (classification_report, confusion_matrix, ConfusionMatrixDisplay,
                             roc_auc_score, average_precision_score, precision_recall_curve)
from sklearn.inspection import permutation_importance

pd.set_option("display.float_format", lambda x: f"{x:,.2f}")  # números con separador de miles

In [ ]:
# ============================================================
# 2. GENERAR LOS DATOS ALEATORIOS (VERSIÓN REALISTA)
# ============================================================
# Una versión "ingenua" sería sortear todo con números uniformes y asignar el fraude
# al azar: eso NO es realista porque el fraude no tendría ningún patrón.
# Por eso aquí se aplican estas ideas del mundo real:
#   - Los ingresos NO son uniformes: pocos ganan mucho (distribución log-normal).
#   - El monto de una compra depende del ingreso de la persona.
#   - La mayoría de compras ocurren de día y cerca de casa.
#   - El fraude es raro (~3 %) y es más probable de madrugada, lejos de casa
#     y con montos altos respecto al ingreso.

rng = np.random.default_rng(42)   # semilla: hace que los resultados sean reproducibles
N = 20_000                        # número de transacciones

# --- Ingreso mensual (COP): log-normal, mediana ≈ 2.8 millones ---
ingreso = rng.lognormal(mean=np.log(2_800_000), sigma=0.55, size=N)
ingreso = np.clip(ingreso, 1_400_000, 25_000_000)   # mínimo ≈ salario mínimo, máximo razonable
ingreso = np.round(ingreso, -4)                     # redondeo a decenas de miles

# --- Hora del día: más actividad en horas de la tarde, poca de madrugada ---
horas = np.arange(24)
peso_hora = 0.15 + np.exp(-0.5 * ((horas - 14) / 4.5) ** 2)   # campana centrada a las 2 p.m.
peso_hora = peso_hora / peso_hora.sum()                       # se vuelve probabilidad (suma 1)
hora = rng.choice(horas, size=N, p=peso_hora) + rng.random(N) # hora + minutos (decimal)
hora = np.minimum(np.round(hora, 2), 23.99)   # evita que redondee a 24.00

# --- Distancia a la residencia (km): 80 % compras cercanas, 20 % lejanas (viajes, otra ciudad) ---
cerca = rng.exponential(scale=6, size=N)
lejos = rng.lognormal(mean=3.2, sigma=0.8, size=N)
distancia = np.where(rng.random(N) < 0.80, cerca, lejos)
distancia = np.round(distancia, 1)

# --- Monto de la compra (COP): proporcional al ingreso (≈2 % del ingreso, con mucha variación) ---
monto = ingreso * rng.lognormal(mean=np.log(0.02), sigma=0.9, size=N)
monto = np.clip(monto, 5_000, None)    # compra mínima de 5.000 pesos
monto = np.round(monto, -2)            # redondeo a centenas de pesos

# --- Probabilidad de fraude: modelo logístico con "factores de riesgo" ---
es_madrugada = ((hora >= 0) & (hora < 5)).astype(int)        # 12 a.m. - 5 a.m.
ratio = monto / ingreso                                       # qué tan grande es la compra vs. el ingreso
riesgo = (1.6 * es_madrugada                                  # madrugada: mucho más riesgo
          + 0.8 * np.log1p(distancia)                         # más lejos: más riesgo
          + 0.9 * np.clip(np.log(ratio / 0.02), -3, 4))       # monto alto vs. ingreso: más riesgo

# Se busca el intercepto para que el fraude sea ≈3 % del total (desbalance real)
def sigmoide(z):
    return 1 / (1 + np.exp(-z))

lo, hi = -20, 5
for _ in range(60):                       # búsqueda binaria
    medio = (lo + hi) / 2
    if sigmoide(medio + riesgo).mean() > 0.03:
        hi = medio
    else:
        lo = medio
prob_fraude = sigmoide(medio + riesgo)
fraude = (rng.random(N) < prob_fraude).astype(int)   # 1 = fraude, 0 = legítima (con algo de azar)

# --- Armar la tabla ---
df = pd.DataFrame({
    "monto_cop": monto,
    "hora_del_dia": hora,
    "distancia_residencia_km": distancia,
    "ingreso_mensual_cop": ingreso,
    "fraude": np.where(fraude == 1, "Sí", "No"),
})

# --- Guardar el archivo (CSV) ---
df.to_csv("transacciones_fraude.csv", index=False)
print("Archivo 'transacciones_fraude.csv' creado con", len(df), "filas")
df.head(10)

In [ ]:
# ============================================================
# 3. REVISIÓN DE LOS DATOS (¿SE VEN REALES?)
# ============================================================
print("Porcentaje de fraude:")
print(df["fraude"].value_counts(normalize=True).mul(100).round(2).astype(str) + " %")
print("\nResumen estadístico:")
display(df.describe())

# Comparar el promedio de cada variable entre fraudes y no fraudes
df["fraude_num"] = (df["fraude"] == "Sí").astype(int)
display(df.groupby("fraude")[["monto_cop", "hora_del_dia", "distancia_residencia_km", "ingreso_mensual_cop"]].median())

# Gráficas rápidas
fig, ax = plt.subplots(1, 3, figsize=(16, 4))
df.groupby(df["hora_del_dia"].astype(int))["fraude_num"].mean().mul(100).plot(kind="bar", ax=ax[0], color="tomato")
ax[0].set_title("% de fraude por hora"); ax[0].set_xlabel("Hora"); ax[0].set_ylabel("% fraude")
ax[1].hist(np.log10(df["monto_cop"]), bins=40, color="steelblue")
ax[1].set_title("Distribución del monto (log10 COP)")
ax[2].hist(df["distancia_residencia_km"].clip(upper=100), bins=40, color="seagreen")
ax[2].set_title("Distancia a la residencia (km, tope 100)")
plt.tight_layout(); plt.show()

In [ ]:
# ============================================================
# 4. PREPARAR LAS VARIABLES PARA EL MODELO
# ============================================================
d = df.copy()
# La hora es circular (23:59 está cerca de 00:00), por eso se usan seno y coseno
d["hora_sin"] = np.sin(2 * np.pi * d["hora_del_dia"] / 24)
d["hora_cos"] = np.cos(2 * np.pi * d["hora_del_dia"] / 24)
d["es_madrugada"] = ((d["hora_del_dia"] >= 0) & (d["hora_del_dia"] < 5)).astype(int)
# Variables derivadas que ayudan al modelo (feature engineering)
d["monto_sobre_ingreso"] = d["monto_cop"] / d["ingreso_mensual_cop"]
d["log_distancia"] = np.log1p(d["distancia_residencia_km"])

variables = ["monto_cop", "hora_sin", "hora_cos", "es_madrugada",
             "distancia_residencia_km", "log_distancia",
             "ingreso_mensual_cop", "monto_sobre_ingreso"]
X = d[variables]            # entradas
y = d["fraude_num"]         # salida: 1 = fraude

# Separar entrenamiento (80 %) y prueba (20 %). 'stratify' mantiene el mismo % de fraude en ambos.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=42)
print("Entrenamiento:", X_train.shape, "| Prueba:", X_test.shape)

In [ ]:
# ============================================================
# 5. ENTRENAR Y COMPARAR MODELOS
# ============================================================
# class_weight="balanced": como hay muy pocos fraudes, se les da más peso al entrenar
# para que el modelo no aprenda a decir siempre "no es fraude".
modelos = {
    "Regresión logística": make_pipeline(StandardScaler(),
                                         LogisticRegression(class_weight="balanced", max_iter=1000)),
    "Random Forest": RandomForestClassifier(n_estimators=300, min_samples_leaf=5,
                                            class_weight="balanced", n_jobs=-1, random_state=42),
    "Gradient Boosting": HistGradientBoostingClassifier(class_weight="balanced", random_state=42),
}

resultados = []
for nombre, modelo in modelos.items():
    modelo.fit(X_train, y_train)                      # entrenar
    prob = modelo.predict_proba(X_test)[:, 1]         # probabilidad de fraude en datos de prueba
    resultados.append({
        "Modelo": nombre,
        "ROC-AUC": roc_auc_score(y_test, prob),             # capacidad general de separar clases
        "PR-AUC": average_precision_score(y_test, prob),    # mejor métrica cuando hay pocos fraudes
    })

tabla = pd.DataFrame(resultados).sort_values("PR-AUC", ascending=False).reset_index(drop=True)
display(tabla)
mejor_nombre = tabla.loc[0, "Modelo"]
mejor_modelo = modelos[mejor_nombre]
print("Mejor modelo según PR-AUC:", mejor_nombre)

In [ ]:
# ============================================================
# 6. ELEGIR EL UMBRAL DE DECISIÓN
# ============================================================
# Por defecto se marca fraude si probabilidad > 0.5, pero con clases desbalanceadas
# casi nunca es lo óptimo. Se busca el umbral que maximiza el F1 usando validación cruzada
# SOLO con datos de entrenamiento (así no se "hace trampa" con los datos de prueba).
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
prob_cv = cross_val_predict(mejor_modelo, X_train, y_train, cv=cv, method="predict_proba")[:, 1]
precision, recall, umbrales = precision_recall_curve(y_train, prob_cv)
f1 = 2 * precision * recall / (precision + recall + 1e-9)
umbral = umbrales[np.argmax(f1[:-1])]
print(f"Umbral elegido: {umbral:.3f}")

In [ ]:
# ============================================================
# 7. EVALUACIÓN FINAL EN DATOS DE PRUEBA
# ============================================================
prob_test = mejor_modelo.predict_proba(X_test)[:, 1]
pred_test = (prob_test >= umbral).astype(int)

print(classification_report(y_test, pred_test, target_names=["No fraude", "Fraude"]))
# precision: de los que marcó como fraude, cuántos realmente lo eran
# recall:    de los fraudes reales, cuántos logró detectar

ConfusionMatrixDisplay(confusion_matrix(y_test, pred_test),
                       display_labels=["No fraude", "Fraude"]).plot(cmap="Blues")
plt.title("Matriz de confusión (datos de prueba)"); plt.show()

In [ ]:
# ============================================================
# 8. ¿QUÉ VARIABLES IMPORTAN MÁS?
# ============================================================
# Permutation importance: se mezclan los valores de una variable y se mide cuánto empeora el modelo.
imp = permutation_importance(mejor_modelo, X_test, y_test, scoring="average_precision",
                             n_repeats=10, random_state=42)
(pd.Series(imp.importances_mean, index=variables).sort_values()
   .plot(kind="barh", figsize=(7, 4), color="darkorange", title="Importancia de variables"))
plt.xlabel("Caída en PR-AUC al mezclar la variable"); plt.show()

In [ ]:
# ============================================================
# 9. USAR EL MODELO CON UNA TRANSACCIÓN NUEVA
# ============================================================
def evaluar_transaccion(monto_cop, hora, distancia_km, ingreso_cop):
    """Devuelve la probabilidad de fraude y la decisión del modelo."""
    h = hora
    fila = pd.DataFrame([{
        "monto_cop": monto_cop,
        "hora_sin": np.sin(2 * np.pi * h / 24),
        "hora_cos": np.cos(2 * np.pi * h / 24),
        "es_madrugada": int(0 <= h < 5),
        "distancia_residencia_km": distancia_km,
        "log_distancia": np.log1p(distancia_km),
        "ingreso_mensual_cop": ingreso_cop,
        "monto_sobre_ingreso": monto_cop / ingreso_cop,
    }])[variables]
    p = mejor_modelo.predict_proba(fila)[0, 1]
    return p, ("⚠️ POSIBLE FRAUDE" if p >= umbral else "✅ Parece legítima")

# Ejemplo 1: compra normal (almuerzo, 1 p.m., cerca de casa)
print(evaluar_transaccion(35_000, 13, 2, 3_000_000))
# Ejemplo 2: compra grande, 3 a.m., a 150 km de casa
print(evaluar_transaccion(1_800_000, 3, 150, 3_000_000))

In [ ]:
# ============================================================
# 10. (OPCIONAL) DESCARGAR EL ARCHIVO CSV GENERADO
# ============================================================
from google.colab import files
files.download("transacciones_fraude.csv")